In [67]:
import h5py
import numpy as np

In [68]:
f=h5py.File("/srv/scratch/anusri/model/h3k4me3/4K_out_gm12878.h3k4me3.seed.2345.cs.25.filters.500.naive.range.4.6.to.11.5.0.predictions",'r')
#f=h5py.File("/srv/scratch/anusri/model/h3k27ac_params/5K_out_9_gm12878.h3k27ac.seed.2345.cs.25.filters.500.naive.range.4.6.to.11.5.0.predictions",'r')
#f=h5py.File("/srv/scratch/anusri/model/h3k27ac_params/2nd_patience_6_5K_out_9_gm12878.h3k27ac.seed.2345.cs.25.filters.300.naive.range.4.6.to.11.5.0.predictions",'r')
#f=h5py.File("/srv/scratch/anusri/model/h3k27ac/5K_out_10_gm12878.h3k27ac.seed.1234.cs.25.filters.500.naive.range.4.6.to.11.5.0.predictions",'r')

OSError: Unable to open file (unable to open file: name = '/srv/scratch/anusri/model/h3k4me3/4K_out_gm12878.h3k4me3.seed.2345.cs.25.filters.500.naive.range.4.6.to.11.5.0.predictions', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
labels_0=f['lab_0'][:]
labels_1=f['lab_1'][:]
pred_0=f['pred_0'][:]
pred_1=f['pred_1'][:]
coord=f['coords'][:]

In [ ]:
coord_fixed=[[i.decode('utf8')  for i in j] for j in coord]

In [ ]:
outf=open("tmp.txt",'w')
for entry in coord_fixed: 
    outf.write(str(entry)+'\n')
outf.close()

In [ ]:
coord_dict={} 
for i in range(len(coord_fixed)): 
    coord_dict[tuple(coord_fixed[i])]=i

In [ ]:
import pandas as pd 
from scipy.stats import spearmanr 
from scipy.stats import pearsonr 
from scipy.special import softmax

get ranked list of IDR peaks on chrom 1 (test chrom)

In [ ]:
idr_peaks=pd.read_csv('/mnt/lab_data3/anusri/histone_data/processed_data/croo/histone_H3K4me3/peak/overlap_reproducibility/test.bed.gz',header=None,sep='\t')
idr_peaks['summit']=idr_peaks[1]+idr_peaks[9]
idr_peaks=idr_peaks.sort_values(by=[8],ascending=False)
positions=[]
top_peaks=20
count=0
for index,row in idr_peaks.iterrows(): 
    summit=row['summit']
    chrom=row[0]
    count+=1
    if count >top_peaks: 
        break
    positions.append((chrom,str(summit),'.'))

In [ ]:
positions[0]

In [ ]:
#coord_dict[positions[0]]

In [ ]:
## for plotting 
import matplotlib 
from matplotlib import pyplot as plt
plt.rcParams["figure.figsize"]=20,5

In [ ]:
font = {'family' : 'normal',
        'weight' : 'bold',
        'size'   : 25}

matplotlib.rc('font', **font)

In [ ]:
plt.plot(softmax(pred_0[0,:,0]))
#plt.plot(pred_0_softmax[0,:,0])

In [ ]:
pred_0_softmax=softmax(pred_0,axis=1)

In [ ]:
pred_0_softmax.shape

In [ ]:
plt.plot(pred_0_softmax[0,:,0]*pred_1[0,0])
plt.plot(pred_0_softmax[0,:,1]*pred_1[0,1])

In [ ]:
plt.rcParams["figure.figsize"]=40,25
title="H3K4me1, GM12878, test set, fold 0"
for pos in positions[0:20]: 
    cur_index=coord_dict[pos]
    print(cur_index)
    counts_plus=pred_0_softmax[cur_index,:,0]*pred_1[cur_index,0]
    counts_minus=pred_0_softmax[cur_index,:,1]*pred_1[cur_index,1]
    prob_plus=pred_0_softmax[cur_index,:,0]
    prob_minus=pred_0_softmax[cur_index,:,1]
    fig, axes = plt.subplots(3, 1)
    probs_observed_forward=labels_0[cur_index,:,0]/sum(labels_0[cur_index,:,0])
    probs_observed_reverse=labels_0[cur_index,:,1]/sum(labels_0[cur_index,:,1])
    #axes[0].plot(counts_plus,label='Predictions, + strand',color='r')
    axes[1].plot(labels_0[cur_index,:,0],label='Labels, + strand',color='k')
    axes[2].plot(probs_observed_forward,label="Labels Probability, + strand",color='k', alpha=0.5)
    axes[2].plot(pred_0_softmax[cur_index,:,0],label="Predicted Probability,+ strand",color='r', alpha=0.5)    
    axes[0].plot(-1*counts_minus,label='Predictions, - strand',color='b')
    axes[1].plot(-1*labels_0[cur_index,:,1],label='Labels, - strand',color='g')
    axes[2].plot(-1*probs_observed_reverse,label="Labels Probability, - strand",color='g', alpha=0.5)
    axes[2].plot(-1*pred_0_softmax[cur_index,:,1],label="Predicted Probability,- strand",color='b', alpha=0.5)            
    axes[0].set_title(title+str(pos))
    axes[1].set_title(title+str(pos))
    axes[2].set_title(title+str(pos))
    axes[0].legend(loc="upper right")
    axes[1].legend(loc="upper right") 
    axes[2].legend(loc="upper right")
    fig.tight_layout()
    plt.show()
    #bins = np.linspace(0, 0.002, 30)
    #print("observed")
    #print(np.sum(sum(labels_0[cur_index,:,0])))
    #print(probs_observed_forward)
    #print(np.sum(probs_observed_forward))
    #print(np.min(probs_observed_forward), np.max(probs_observed_forward), np.mean(probs_observed_forward), np.std(probs_observed_forward))
    #print("predicted")
    #print(pred_0_softmax[cur_index,:,0])
    #print(np.sum(pred_0_softmax[cur_index,:,0]))
    #print(np.min(pred_0_softmax[cur_index,:,0]), np.max(pred_0_softmax[cur_index,:,0]), np.mean(pred_0_softmax[cur_index,:,0]), np.std(pred_0_softmax[cur_index,:,0]))
    
    #plt.hist(probs_observed_forward,bins=bins,  label="observed")
    #plt.hist(pred_0_softmax[cur_index,:,0],bins=bins, label="predicted")
    #plt.legend()
    #plt.show()